# Chapter 15 -- Observability, Tracing & Debugging (Your Working Copy)

Work through this notebook **after reading** `notes/ch15-observability.md`. This chapter instruments Chapter 13's orchestrator + 3-subagent multi-agent system with a real, hand-rolled OTel-GenAI-convention span tree (no external collector needed -- spans are plain Python dicts written to a local JSONL file, which is exactly what a real OTel exporter does under the hood), builds a small trace viewer that prints the tree with per-span cost/latency, then reproduces the notes' critical-path / Amdahl's-trap dry-run against the real, actually-measured trace this notebook itself produces.

Two exercises below have a stub to fill in: **trace-to-eval promotion** (notes Section 6) and a **trajectory-diff tool** (notes Section 7). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- A Real Span Recorder, OTel-GenAI-Convention-Shaped (Given)

`SpanRecorder` is a real, working span emitter -- every `start_span`/`end_span` call genuinely measures wall-clock time with `time.perf_counter()` and appends a real dict (matching notes Section 3's `gen_ai.*` attribute names) to an in-memory list, then to a JSONL file, exactly the shape a real OTel exporter writes to disk. No mocking: the timings below are real durations of real (simulated, sleeping) work, not asserted numbers.

In [ ]:
import json
import time
import uuid
import random
from pathlib import Path
from contextlib import contextmanager

TRACE_LOG_PATH = Path("ch15_trace_log.jsonl")


class SpanRecorder:
    """A minimal, real span recorder shaped like the OTel GenAI conventions (notes Section 3)."""

    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.spans = []

    @contextmanager
    def span(self, trace_id, span_id, parent_span_id, name, operation_name, extra_attributes=None):
        start = time.perf_counter()
        record = {
            "trace_id": trace_id,
            "span_id": span_id,
            "parent_span_id": parent_span_id,
            "name": name,
            "attributes": {"gen_ai.operation.name": operation_name, **(extra_attributes or {})},
        }
        try:
            yield record
        finally:
            end = time.perf_counter()
            record["duration_ms"] = round((end - start) * 1000, 2)
            self.spans.append(record)
            with open(self.log_path, "a") as f:
                f.write(json.dumps(record) + "\n")

    def reset_log(self):
        if self.log_path.exists():
            self.log_path.unlink()
        self.spans = []


recorder = SpanRecorder(TRACE_LOG_PATH)
recorder.reset_log()
print(f"SpanRecorder ready, writing to: {TRACE_LOG_PATH.resolve()}")


## Part 2 -- Instrumenting Chapter 13's Orchestrator + 3-Subagent Run (Given)

`run_instrumented_orchestrator` reproduces the exact 12-span tree from notes Section 2: one root `invoke_agent` span, an orchestrator "plan" chat span, 3 concurrent `invoke_agent` subagent spans (each containing a "decide" chat span and a "web_search" tool span, run with real `ThreadPoolExecutor` concurrency and real `time.sleep` latency so the durations are genuinely, not just nominally, real), and a final orchestrator "synthesize" chat span. Token counts on the 5 chat spans match the notes' Section 8-Bis cost-attribution table exactly.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# (sleep_seconds, input_tokens, output_tokens) per chat span -- matches notes Section 8-Bis/11 tables exactly.
CHAT_SPEC = {
    "orchestrator_plan": (0.08, 600, 200),
    "subagent_1_decide": (0.12, 900, 300),
    "subagent_2_decide": (0.06, 500, 250),
    "subagent_3_decide": (0.04, 400, 150),
    "orchestrator_synthesize": (0.10, 1200, 400),
}
# (sleep_seconds) per tool span -- matches notes Section 11's duration table.
TOOL_SPEC = {
    "subagent_1_web_search": 0.09,
    "subagent_2_web_search": 0.15,
    "subagent_3_web_search": 0.07,
}
COST_PER_1K_TOKENS = 0.003


def run_subagent(recorder, trace_id, orchestrator_span_id, subagent_name):
    """One subagent's real work: a chat 'decide' span, then a tool 'web_search' span, genuinely sequential."""
    subagent_span_id = str(uuid.uuid4())[:8]
    with recorder.span(trace_id, subagent_span_id, orchestrator_span_id, f"invoke_agent:{subagent_name}", "invoke_agent"):
        decide_key = f"{subagent_name}_decide"
        sleep_s, in_tok, out_tok = CHAT_SPEC[decide_key]
        chat_span_id = str(uuid.uuid4())[:8]
        with recorder.span(trace_id, chat_span_id, subagent_span_id, f"chat:{decide_key}", "chat",
                            {"gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok}):
            time.sleep(sleep_s)

        tool_key = f"{subagent_name}_web_search"
        tool_sleep = TOOL_SPEC[tool_key]
        tool_span_id = str(uuid.uuid4())[:8]
        with recorder.span(trace_id, tool_span_id, subagent_span_id, f"execute_tool:{tool_key}", "execute_tool",
                            {"gen_ai.tool.name": "web_search"}):
            time.sleep(tool_sleep)
    return subagent_span_id


def run_instrumented_orchestrator(recorder):
    """Reproduces notes Section 2's 12-span tree with REAL measured concurrency and latency."""
    trace_id = str(uuid.uuid4())[:8]
    root_span_id = str(uuid.uuid4())[:8]

    with recorder.span(trace_id, root_span_id, None, "invoke_agent:orchestrator", "invoke_agent"):
        sleep_s, in_tok, out_tok = CHAT_SPEC["orchestrator_plan"]
        plan_span_id = str(uuid.uuid4())[:8]
        with recorder.span(trace_id, plan_span_id, root_span_id, "chat:orchestrator_plan", "chat",
                            {"gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok}):
            time.sleep(sleep_s)

        with ThreadPoolExecutor(max_workers=3) as pool:
            list(pool.map(
                lambda name: run_subagent(recorder, trace_id, root_span_id, name),
                ["subagent_1", "subagent_2", "subagent_3"],
            ))

        sleep_s, in_tok, out_tok = CHAT_SPEC["orchestrator_synthesize"]
        synth_span_id = str(uuid.uuid4())[:8]
        with recorder.span(trace_id, synth_span_id, root_span_id, "chat:orchestrator_synthesize", "chat",
                            {"gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok}):
            time.sleep(sleep_s)

    return trace_id


trace_id = run_instrumented_orchestrator(recorder)
print(f"Recorded trace_id={trace_id!r} with {len(recorder.spans)} spans.")
assert len(recorder.spans) == 12, f"expected 12 spans, got {len(recorder.spans)}"
print("PASS -- 12 spans recorded, matching notes Section 2's tree exactly.")


## Part 3 -- A Small Trace Viewer: Print the Tree With Per-Span Cost/Latency (Given)

`print_trace_tree` does exactly what notes Section 2 says a trace viewer's job mechanically is: group spans by `trace_id`, link each under its `parent_span_id`, and walk the resulting tree depth-first, printing duration and (for chat spans) cost at every node.

In [ ]:
def build_tree(spans, trace_id):
    """Group by trace_id, link by parent_span_id -- the mechanical trace-viewer job from notes Section 2."""
    by_id = {s["span_id"]: s for s in spans if s["trace_id"] == trace_id}
    children = {}
    root = None
    for s in by_id.values():
        parent = s["parent_span_id"]
        if parent is None:
            root = s
        else:
            children.setdefault(parent, []).append(s)
    return root, children


def span_cost(span):
    attrs = span["attributes"]
    if "gen_ai.usage.input_tokens" not in attrs:
        return 0.0
    tokens = attrs["gen_ai.usage.input_tokens"] + attrs["gen_ai.usage.output_tokens"]
    return round(tokens / 1000 * COST_PER_1K_TOKENS, 5)


def print_trace_tree(spans, trace_id):
    root, children = build_tree(spans, trace_id)

    def walk(span, depth):
        cost = span_cost(span)
        cost_str = f" cost=${cost:.5f}" if cost else ""
        print(f"{'  ' * depth}{span['name']:40s} {span['duration_ms']:>8.2f} ms{cost_str}")
        for child in children.get(span["span_id"], []):
            walk(child, depth + 1)

    print(f"Trace {trace_id}:")
    walk(root, 0)


print_trace_tree(recorder.spans, trace_id)


## Part 4 -- Cost and Latency Rollups (Given)

Rolling per-subagent and total cost/latency up from the printed tree above -- notes Section 8's attribution chain, computed from the real spans this notebook just produced (not the notes' illustrative fixed numbers, though they should be close, modulo real measured latency noise).

In [ ]:
def rollup_by_subtree(spans, trace_id):
    """notes Section 8 -- roll up total cost per top-level subtree (orchestrator's own spans vs each subagent)."""
    root, children = build_tree(spans, trace_id)
    by_id = {s["span_id"]: s for s in spans if s["trace_id"] == trace_id}

    def descendants(span_id):
        result = [by_id[span_id]]
        for child in children.get(span_id, []):
            result.extend(descendants(child["span_id"]))
        return result

    buckets = {}
    orchestrator_own = [root] + [c for c in children.get(root["span_id"], []) if not c["name"].startswith("invoke_agent")]
    buckets["orchestrator"] = sum(span_cost(s) for s in orchestrator_own)
    for child in children.get(root["span_id"], []):
        if child["name"].startswith("invoke_agent"):
            buckets[child["name"]] = sum(span_cost(s) for s in descendants(child["span_id"]))
    return buckets


rollup = rollup_by_subtree(recorder.spans, trace_id)
total_cost = sum(rollup.values())
print("Per-subtree cost rollup:")
for name, cost in rollup.items():
    pct = (cost / total_cost * 100) if total_cost else 0
    print(f"  {name:30s} ${cost:.5f}  ({pct:.1f}%)")
print(f"\nTotal trace cost: ${total_cost:.5f}")


## Part 5 -- Exercise 1: Trace-to-Eval Promotion (notes Section 6)

Simulate the flywheel: a batch of production traces comes in, some pass their verifier check and some don't. `promote_failed_traces_to_golden_set` should scan a batch of traces, and for every one whose `verifier_outcome` is `"fail"`, build a new golden-dataset entry (a dict with the trace's `trace_id`, its root span's task description, and the verifier's recorded reason) -- exactly notes Section 6's "promote a failed production trace into a golden eval case."

Fill in `promote_failed_traces_to_golden_set(production_traces)` below.

In [ ]:
PRODUCTION_TRACES = [
    {"trace_id": "p1", "task": "summarize Q3 earnings call", "verifier_outcome": "pass", "reason": None},
    {"trace_id": "p2", "task": "extract action items from meeting notes", "verifier_outcome": "fail",
     "reason": "missed an action item buried in a bulleted aside"},
    {"trace_id": "p3", "task": "classify support ticket urgency", "verifier_outcome": "pass", "reason": None},
    {"trace_id": "p4", "task": "reconcile two CSV exports", "verifier_outcome": "fail",
     "reason": "silently dropped rows with a trailing comma"},
    {"trace_id": "p5", "task": "draft a refund policy explanation", "verifier_outcome": "pass", "reason": None},
]


def promote_failed_traces_to_golden_set(production_traces):
    """notes Section 6 -- turn every verifier-failed trace into a golden-dataset candidate entry."""
    # TODO: for each trace in production_traces where trace["verifier_outcome"] == "fail",
    # build a dict {"source_trace_id": trace["trace_id"], "prompt": trace["task"],
    # "known_failure_reason": trace["reason"]} and collect all of them into a list.
    # Return that list (traces that passed are NOT promoted).
    return []


promoted = promote_failed_traces_to_golden_set(PRODUCTION_TRACES)
print(f"Promoted {len(promoted)} failed production trace(s) into golden-dataset candidates:")
for entry in promoted:
    print(f"  {entry}")

assert len(promoted) == 2, f"expected exactly 2 promoted entries (p2, p4), got {len(promoted)}"
assert {e['source_trace_id'] for e in promoted} == {"p2", "p4"}
print("\nPASS -- only the 2 verifier-failed traces were promoted; the 3 passing traces were correctly skipped.")


## Part 6 -- Exercise 2: A Trajectory-Diff Tool (notes Section 7)

Build a second, deliberately-different trace (a "yesterday" run where subagent_2's web search took much longer and subagent_2's chat span used more tokens -- a plausible harness regression) and implement `diff_traces(spans_a, spans_b, trace_id_a, trace_id_b)`: walk both trees position-by-position, over **leaf spans only** (`chat`/`execute_tool` -- an `invoke_agent` container's own duration mechanically inherits any change in its children, so comparing containers would only ever tell you "somewhere under here changed," never which leaf), and return the **first** span position where the two traces' span `name` differs, OR -- if the names match at every position -- the first position where `duration_ms` differs by more than a tolerance, matching notes Section 7's "stop at the first structural divergence" principle.

Fill in `diff_traces` below.

In [ ]:
recorder_b = SpanRecorder(Path("ch15_trace_log_b.jsonl"))
recorder_b.reset_log()

# Monkey-patch a slower subagent_2 web_search + a bigger subagent_2 chat call for the "yesterday" comparison trace.
CHAT_SPEC["subagent_2_decide"] = (0.06, 500, 900)  # output tokens ballooned: 250 -> 900
TOOL_SPEC["subagent_2_web_search"] = 0.30           # tool latency ballooned: 0.15s -> 0.30s
trace_id_b = run_instrumented_orchestrator(recorder_b)
# restore for anyone re-running this notebook top to bottom without a kernel restart
CHAT_SPEC["subagent_2_decide"] = (0.06, 500, 250)
TOOL_SPEC["subagent_2_web_search"] = 0.15

print(f"Trace A (today):     {trace_id}")
print(f"Trace B (yesterday): {trace_id_b}")


def flatten_depth_first(spans, trace_id):
    """Depth-first flattening of the LEAF spans only (chat/execute_tool, not invoke_agent containers),
    so 'position i in trace A' lines up with 'position i in trace B', and a container span's duration
    (which mechanically inherits any change in its children) never masks WHICH leaf actually changed."""
    root, children = build_tree(spans, trace_id)
    order = []

    def walk(span):
        kids = sorted(children.get(span["span_id"], []), key=lambda s: s["name"])
        if not kids:
            order.append(span)
        for child in kids:
            walk(child)

    walk(root)
    return order


def diff_traces(spans_a, spans_b, trace_id_a, trace_id_b, duration_tolerance_ms=50):
    """notes Section 7 -- walk both trees in the same order; return the FIRST position that diverges."""
    # TODO: flatten_depth_first(spans_a, trace_id_a) and flatten_depth_first(spans_b, trace_id_b)
    # into order_a / order_b (leaf spans only). Walk them together by index. At the first index i
    # where order_a[i]["name"] != order_b[i]["name"], return {"index": i, "kind": "structural",
    # "a": order_a[i]["name"], "b": order_b[i]["name"]}.
    # If every name matches, find the first index where
    # abs(order_a[i]["duration_ms"] - order_b[i]["duration_ms"]) > duration_tolerance_ms and return
    # {"index": i, "kind": "duration", "span": order_a[i]["name"],
    #  "duration_a": order_a[i]["duration_ms"], "duration_b": order_b[i]["duration_ms"]}.
    # If nothing diverges at all, return None.
    return None


divergence = diff_traces(recorder.spans, recorder_b.spans, trace_id, trace_id_b)
print(f"\nFirst divergence found: {divergence}")

assert divergence is not None, "expected a real divergence -- subagent_2's chat and tool spans were deliberately changed"
assert divergence["kind"] == "duration", f"expected a duration divergence (names are identical), got {divergence['kind']}"
assert "subagent_2" in divergence["span"], f"expected the first divergence to be in subagent_2's branch, got {divergence['span']}"
print("PASS -- correctly localized the regression to subagent_2's branch, at the first point the two traces diverge.")


## Part 7 -- Reproducing the Notes Section 11 Dry-Run Against This Notebook's OWN Real Trace (Given)

The notes' critical-path / Amdahl's-trap dry-run used illustrative fixed durations. Here we recompute the exact same critical path, parallelizable fraction, and Amdahl comparison against the **real, measured** durations `recorder.spans` produced above -- so this is not a re-assertion of the notes' numbers, it's an independent recomputation on real (if noisy) wall-clock data, which should land close to the notes' structure even though the exact milliseconds differ run to run.

In [ ]:
def compute_critical_path_and_amdahl(spans, trace_id):
    root, children = build_tree(spans, trace_id)
    by_name = {s["name"]: s for s in spans if s["trace_id"] == trace_id}

    plan = by_name["chat:orchestrator_plan"]["duration_ms"]
    synth = by_name["chat:orchestrator_synthesize"]["duration_ms"]
    subagent_totals = {}
    for sub_name in ("subagent_1", "subagent_2", "subagent_3"):
        sub_span = next(c for c in children[root["span_id"]] if c["name"] == f"invoke_agent:{sub_name}")
        subagent_totals[sub_name] = sub_span["duration_ms"]

    critical_path = plan + max(subagent_totals.values()) + synth
    sequential_baseline = plan + sum(subagent_totals.values()) + synth
    parallel_work = sum(subagent_totals.values())
    p = parallel_work / sequential_baseline
    amdahl_infinite = 1 / (1 - p)
    real_best_case_speedup = sequential_baseline / critical_path

    return {
        "subagent_totals_ms": subagent_totals,
        "critical_path_ms": round(critical_path, 2),
        "sequential_baseline_ms": round(sequential_baseline, 2),
        "parallelizable_fraction": round(p, 4),
        "amdahl_infinite_processor_speedup": round(amdahl_infinite, 3),
        "real_best_case_speedup": round(real_best_case_speedup, 3),
    }


result = compute_critical_path_and_amdahl(recorder.spans, trace_id)
print("Recomputed from THIS notebook's own real, measured trace:")
for k, v in result.items():
    print(f"  {k:32s}: {v}")

print(f"\nAmdahl's classical formula says {result['amdahl_infinite_processor_speedup']:.2f}x is achievable;")
print(f"the real critical-path-bounded ceiling is only {result['real_best_case_speedup']:.2f}x --")
print("exactly the gap notes Section 11 predicts for task-parallel (not data-parallel) subagent fan-out.")
assert result["amdahl_infinite_processor_speedup"] > result["real_best_case_speedup"], \
    "Amdahl's infinite-processor number should overstate the real, critical-path-bounded speedup"
print("\nPASS -- Amdahl overstates the real ceiling on our own measured trace, matching the notes' argument.")


## Optional -- Real-Model Span (Off by Default)

`RUN_REAL_SPAN_DEMO` defaults to `False` so this notebook never makes a real Bedrock call automatically. Flip it to `True` and re-run this cell only if you have real credentials in `.env` and want to see one real `chat` span recorded around an actual model call.

In [ ]:
RUN_REAL_SPAN_DEMO = False

if RUN_REAL_SPAN_DEMO and AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    real_trace_id = str(uuid.uuid4())[:8]
    with recorder.span(real_trace_id, str(uuid.uuid4())[:8], None, "chat:real_model_call", "chat") as span:
        response = real_client.messages.create(
            model=MODEL_NAME, max_tokens=20,
            messages=[{"role": "user", "content": "Say hello in exactly three words."}],
        )
        span["attributes"]["gen_ai.usage.input_tokens"] = response.usage.input_tokens
        span["attributes"]["gen_ai.usage.output_tokens"] = response.usage.output_tokens
    print_trace_tree(recorder.spans, real_trace_id)
else:
    print("Skipped -- set RUN_REAL_SPAN_DEMO=True and provide real .env credentials to run this cell for real.")
